In [1]:
%display latex

# Rearranging the Equations of Motion as a Vector ODE
Using Lagrangian mechanics, we can use the Euler-Lagrange equation to derive the following equations of motion for the double pendulum.
$$
    l_1 \ddot\theta_1 + \frac{m_2}{M} l_2 \left[\ddot\theta_2 \cos\alpha - \dot\theta_2^2 \sin\alpha\right] - g\sin\theta_1 = 0
$$
and
$$
    l_2 \ddot\theta_2 + l_1 \left[\ddot\theta_1 \cos\alpha - \dot\theta_1^2 \sin\alpha\right] - g\sin\theta_2 = 0
$$
where $M = m_1 + m_2$ and $\alpha = \theta_1 - \theta_2$.

We wish to apply typical IVP solvers (e.g. Forward Euler, Runge-Kutta methods) which are designed for first order IVPs. Thus we must rearrange this system as a vector ODE.

## Setup
A good first step is to have the equations written symbolically using SageMath

In [20]:
# Symbols -- SageMath can do "derivatives", but we want to take advantage of symbolic solutions, so we don't use this functionality.
var("l_1", "l_2", "m_1", "m_2", "M", "alpha", "g", "theta_1", "theta_2")
var("theta_1_dot", latex_name=r"\dot\theta_1")
var("theta_1_ddot", latex_name=r"\ddot\theta_1")
var("theta_2_dot", latex_name=r"\dot\theta_2")
var("theta_2_ddot", latex_name=r"\ddot\theta_2")
t = var("t")

In [21]:
eq_1 = l_1 * theta_1_ddot + m_2/M * l_2 * (theta_2_ddot * cos(alpha) - theta_2_dot^2 * sin(alpha)) - g * sin(theta_1) == 0
eq_2 = l_2 * theta_2_ddot + l_1 * (theta_1_ddot * cos(alpha) - theta_1_dot^2 * sin(alpha)) - g * sin(theta_2) == 0

In [22]:
eq_1

-(theta_2_dot^2*sin(alpha) - theta_2_ddot*cos(alpha))*l_2*m_2/M + l_1*theta_1_ddot - g*sin(theta_1) == 0

In [23]:
eq_2

-(theta_1_dot^2*sin(alpha) - theta_1_ddot*cos(alpha))*l_1 + l_2*theta_2_ddot - g*sin(theta_2) == 0

## $\ddot\theta$ Equations
The next step is to write $\ddot\theta_k = f_k(\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)$ for $k=1,2$. To do this, realise that the system introduced above is linear in $\ddot\theta_1$ and $\ddot\theta_2$. Therefore, we can apply standard methods.

In [43]:
solutions = solve([eq_1, eq_2], [theta_1_ddot, theta_2_ddot])
theta_1_ddot_eq = solutions[0][0].simplify_full()
theta_2_ddot_eq = solutions[0][1].simplify_full()

In [44]:
theta_1_ddot_eq

theta_1_ddot == (l_1*m_2*theta_1_dot^2*cos(alpha)*sin(alpha) - l_2*m_2*theta_2_dot^2*sin(alpha) + g*m_2*cos(alpha)*sin(theta_2) - M*g*sin(theta_1))/(l_1*m_2*cos(alpha)^2 - M*l_1)

In [45]:
theta_2_ddot_eq

theta_2_ddot == (l_2*m_2*theta_2_dot^2*cos(alpha)*sin(alpha) - M*l_1*theta_1_dot^2*sin(alpha) + M*g*cos(alpha)*sin(theta_1) - M*g*sin(theta_2))/(l_2*m_2*cos(alpha)^2 - M*l_2)

## Vector ODE
Writing $\ddot\theta_k = f_k(\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)$ to be the equations written above for $k = 1,2$, we can write the system as a vector ODE in terms of
$$
    \mathbf{x} = \begin{pmatrix} \theta_1 \\ \theta_2 \\ \dot\theta_1 \\ \dot\theta_2 \end{pmatrix}
$$
Then
$$
    \dot\mathbf{x} = \begin{pmatrix} x_3 \\ x_4 \\ f_1(\mathbf{x}) \\ f_2(\mathbf{x}) \end{pmatrix}
$$
We can apply this to the aforementioned IVP solvers, given that we provide an initial condition $\mathbf{x}^0$.